# Historical Music Theory Query System
## English Sources (TME) — RAG with LangChain, Chroma, and OpenAI

This notebook queries the **Thesaurus Musicarum Enlgicarum (TME)** vector database using a Retrieval-Augmented Generation (RAG) pipeline.

### How it works
1. Your question is matched against ~2000-character text segments stored in a Chroma vector database.
2. The most semantically similar segments are retrieved using cosine similarity on OpenAI embeddings.
3. A GPT-4o-mini model generates an answer grounded in those retrieved passages.

### Usage
- Run all cells in order (or use **Run All**).
- In the **Query** section, set `user_query`, `k` (number of segments), and optional author/date filters.
- Re-run the Query and Display cells to try new questions without reloading the database.

## 1. Imports

In [ ]:
import os

# Disable ChromaDB telemetry before importing chromadb (suppresses noisy capture() errors)
os.environ["ANONYMIZED_TELEMETRY"] = "False"

from datetime import datetime
from io import BytesIO
from typing import List

import pandas as pd
from IPython.display import display, Markdown, HTML
from typing_extensions import TypedDict

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_JUSTIFY
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak

## 2. Configuration

Set your OpenAI API key. The Chroma database path is resolved automatically:

- **On the shared DO droplet (JupyterHub):** set the environment variable `THEORY_LLM_HOME` to the absolute path of the `streamlit-music-theory` directory (e.g. `/home/ubuntu/streamlit-music-theory`). A JupyterHub admin can do this once in the server's environment so all users inherit it.
- **Local development:** if `THEORY_LLM_HOME` is not set, the path falls back to `../chroma_files/` relative to this notebook — which is correct when the notebook is inside `theory_llm/`.

In [ ]:
# --- OpenAI API key ---
# Option A: set as an environment variable before launching Jupyter:
#   export OPENAI_API_KEY="sk-..."
# Option B: uncomment and paste directly (do not commit to git):
# os.environ["OPENAI_API_KEY"] = "sk-..."

openai_api_key = os.environ.get("OPENAI_API_KEY", "")
if not openai_api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set. See instructions above.")

# --- Chroma database path ---
# On the shared droplet an admin sets THEORY_LLM_HOME=/home/ubuntu/streamlit-music-theory
# (or whatever the absolute path is). Falls back to the relative path for local dev.
_base = os.environ.get("THEORY_LLM_HOME", os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
CHROMA_BASE = os.path.join(_base, "chroma_files")

DB_PATH         = os.path.join(CHROMA_BASE, "chroma-db_tme_english")
COLLECTION_NAME = "tme_english"
METADATA_CSV    = os.path.join(os.path.dirname(os.path.abspath("__file__")), "english_html_metadata.csv")

print(f"Chroma DB path : {DB_PATH}")
print(f"Path exists    : {os.path.exists(DB_PATH)}")

## 3. Load the English Vector Store

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    persist_directory=DB_PATH,
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings
)

print(f"Loaded English (TME) database from: {DB_PATH}")

## 4. Browse Available Sources (Optional)

Run this cell to see all authors and titles in the database.

In [ ]:
if os.path.exists(METADATA_CSV):
    sources_df = pd.read_csv(METADATA_CSV)
    display(sources_df[['author', 'title', 'date', 'citation']].drop_duplicates().reset_index(drop=True))
else:
    print(f"Metadata CSV not found at {METADATA_CSV}. Skipping source browse.")

## 5. Helper Functions — Date Range and Author Lists

In [ ]:
def get_date_range(vs):
    """Return (min_date, max_date) rounded to nearest century from the vector store."""
    all_docs = vs.get()
    min_date, max_date = float('inf'), float('-inf')
    for meta in all_docs.get('metadatas', []):
        if not meta:
            continue
        for field in ('date_start', 'date_end'):
            val = meta.get(field)
            if val is not None:
                try:
                    v = int(val)
                    if field == 'date_start':
                        min_date = min(min_date, v)
                    else:
                        max_date = max(max_date, v)
                except (ValueError, TypeError):
                    pass
    if min_date == float('inf'):
        min_date = 500
    if max_date == float('-inf'):
        max_date = 1700
    return (int(min_date) // 100) * 100, ((int(max_date) // 100) + 1) * 100


def get_unique_authors(vs, date_range=None):
    """Return sorted list of unique authors, optionally filtered by date_range=(start, end)."""
    all_docs = vs.get()
    authors = set()
    for meta in all_docs.get('metadatas', []):
        if not meta or 'author' not in meta:
            continue
        if date_range is not None:
            try:
                doc_start = int(meta.get('date_start', 0))
                doc_end   = int(meta.get('date_end',   9999))
            except (ValueError, TypeError):
                doc_start, doc_end = 0, 9999
            if doc_end < date_range[0] or doc_start > date_range[1]:
                continue
        authors.add(meta['author'])
    return sorted(authors)


db_min_date, db_max_date = get_date_range(vector_store)
all_authors = get_unique_authors(vector_store)

print(f"Date range in database: {db_min_date} – {db_max_date}")
print(f"Authors ({len(all_authors)}): {all_authors}")

## 6. Set Filters

Edit the variables below before running the query.

- **`selected_date_range`** — `(start_year, end_year)` tuple, or `None` for all dates.
- **`selected_authors`** — list of author name strings, or `None` / empty list for all authors.
- **`k`** — number of text segments to retrieve (1–20; more = broader context but slower).

In [ ]:
# Date filter — set to None to include all dates
selected_date_range = (db_min_date, db_max_date)   # e.g. (1500, 1700)

# Author filter — set to None or [] to include all authors
# Example: selected_authors = ["Thomas Morley", "Elway Bevin"]
selected_authors = []   # empty = all authors

# Number of segments to retrieve per query
k = 10

# Compute available authors given the current date range filter
available_authors = get_unique_authors(vector_store, date_range=selected_date_range)
effective_authors = selected_authors if selected_authors else available_authors

print(f"Date filter:    {selected_date_range}")
print(f"Authors filter: {'all' if not selected_authors else selected_authors}")
print(f"k (segments):   {k}")
print(f"Authors in range: {len(available_authors)}")

## 7. LLM and RAG Pipeline

In [ ]:
class State(TypedDict):
    question: str
    context: List
    answer: str

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

system_prompt = """You are an expert in historical music theory and musicology.

You are also familiar with medieval Latin, and various early modern forms of English, Italian and French.

Use the following context passages to answer the question.

IMPORTANT: Each text passage is labeled with a Source number (e.g., "Source 1", "Source 2"), author, title, and date.
When citing passages, always reference them by their Source number (e.g., "Source 1", "Source 5") so readers can
find the exact passage. Also mention the author's name when making claims about their ideas.

Include short quotations from the passages to support your statements, with key words from the original text
and translation when appropriate.

If you don't know the answer based on the provided context, say that you don't know.
Use three sentences maximum and keep the answer concise but informative.

Context:
{context}

Question: {question}

Provide a detailed answer with references to specific Source numbers and authors."""

prompt_template = ChatPromptTemplate.from_template(system_prompt)


def retrieve(state: State):
    question = state["question"]
    retriever = vector_store.as_retriever(search_kwargs={"k": k})
    docs = retriever.invoke(question)
    print(f"Retrieved {len(docs)} segments before filtering.")

    # Date filter
    if selected_date_range:
        def in_date_range(doc):
            try:
                doc_start = int(doc.metadata.get('date_start', 0))
                doc_end   = int(doc.metadata.get('date_end',   9999))
            except (ValueError, TypeError):
                return True
            return not (doc_end < selected_date_range[0] or doc_start > selected_date_range[1])
        docs = [d for d in docs if in_date_range(d)]
        print(f"After date filter: {len(docs)} segments.")

    # Author filter
    if selected_authors:
        docs = [d for d in docs if d.metadata.get('author') in selected_authors]
        print(f"After author filter: {len(docs)} segments.")

    return {"context": docs}


def generate_with_author_grouping(state: State):
    docs_with_numbers = list(enumerate(state["context"], 1))

    # Group by author, preserving global source numbers
    author_groups: dict = {}
    for source_num, doc in docs_with_numbers:
        author = doc.metadata.get('author', 'Unknown Author')
        author_groups.setdefault(author, []).append((source_num, doc))

    context_parts = []
    for author, numbered_docs in author_groups.items():
        section = f"\n=== {author} ===\n"
        for source_num, doc in numbered_docs:
            title      = doc.metadata.get('title',      'Unknown Title')
            date       = doc.metadata.get('date',       'Unknown')
            page_range = doc.metadata.get('page_range', 'Unknown')
            section += f"\n[Source {source_num}] '{title}' ({date}), pp. {page_range}:\n{doc.page_content}\n"
        context_parts.append(section)

    formatted_context = "\n".join(context_parts)
    messages = prompt_template.invoke({"context": formatted_context, "question": state["question"]})
    response = llm.invoke(messages)
    return {"answer": response.content}


graph_builder = StateGraph(State).add_sequence([retrieve, generate_with_author_grouping])
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("generate_with_author_grouping", END)
graph = graph_builder.compile()

print("RAG pipeline ready.")

## 8. Query

Edit `user_query` and run this cell. Re-run as many times as you like.

In [ ]:
user_query = "What are the key elements of good music according to the theorists in the database?"

result = graph.invoke({"question": user_query})
print("Done.")

## 9. Display Results

In [ ]:
display(Markdown(f"## Query\n\n{user_query}"))
display(Markdown(f"## Answer\n\n{result['answer']}"))

display(Markdown("---\n## Source Documents"))

for i, doc in enumerate(result['context'], 1):
    meta     = doc.metadata
    author   = meta.get('author',     'Unknown')
    title    = meta.get('title',      'Unknown')
    date_raw = meta.get('date',       'Unknown')
    pages    = meta.get('page_range', 'Unknown')
    citation = meta.get('citation',   'Unknown')

    date_str = str(date_raw)
    if len(date_str) == 4 and date_str.isdigit():
        date = date_raw
    elif 'th' in date_str:
        date = date_str + ' century'
    else:
        date = date_raw

    display(Markdown(
        f"### Source {i}: {author} — *{title}*\n"
        f"**Date:** {date} &nbsp;|&nbsp; **Pages:** {pages}\n\n"
        f"**Citation:** {citation}\n\n"
        f"---\n"
        f"{doc.page_content}"
    ))

## 10. Export to PDF (Optional)

Run this cell to save the last result as a PDF file.

In [ ]:
def create_pdf(question, answer, context_docs, selected_authors_list, date_range=None):
    buffer = BytesIO()
    doc = SimpleDocTemplate(buffer, pagesize=letter,
                            rightMargin=72, leftMargin=72,
                            topMargin=72, bottomMargin=18)
    styles = getSampleStyleSheet()
    title_style   = ParagraphStyle('T', parent=styles['Heading1'],   fontSize=24, spaceAfter=30)
    heading_style = ParagraphStyle('H', parent=styles['Heading2'],   fontSize=14, spaceAfter=12, spaceBefore=12)
    body_style    = ParagraphStyle('B', parent=styles['BodyText'],   fontSize=11, alignment=TA_JUSTIFY, spaceAfter=12)

    elements = []
    elements.append(Paragraph("Historical Music Theory Query Report (TME)", title_style))
    elements.append(Spacer(1, 0.2 * inch))

    date_range_str = f"{date_range[0]} – {date_range[1]}" if date_range else "All Dates"
    authors_str    = ', '.join(selected_authors_list) if selected_authors_list else 'All Authors'
    elements.append(Paragraph(
        f"<b>Date:</b> {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}<br/>"
        f"<b>Database:</b> English (TME)<br/>"
        f"<b>Date Range:</b> {date_range_str}<br/>"
        f"<b>Authors:</b> {authors_str}<br/>"
        f"<b>Segments:</b> {len(context_docs)}",
        body_style
    ))
    elements.append(Spacer(1, 0.3 * inch))

    elements.append(Paragraph("Query", heading_style))
    elements.append(Paragraph(question, body_style))
    elements.append(Spacer(1, 0.2 * inch))

    elements.append(Paragraph("Answer", heading_style))
    for para in answer.split('\n\n'):
        if para.strip():
            elements.append(Paragraph(para, body_style))

    elements.append(PageBreak())
    elements.append(Paragraph("Source Documents", heading_style))

    for i, src in enumerate(context_docs, 1):
        meta = src.metadata
        elements.append(Paragraph(f"<b>Source {i}</b>", heading_style))
        elements.append(Paragraph(
            f"<b>Author:</b> {meta.get('author', 'Unknown')}<br/>"
            f"<b>Title:</b> {meta.get('title', 'Unknown')}<br/>"
            f"<b>Date:</b> {meta.get('date', 'Unknown')}<br/>"
            f"<b>Page:</b> {meta.get('page_range', 'Unknown')}<br/>"
            f"<b>Citation:</b> {meta.get('citation', 'Unknown')}",
            body_style
        ))
        elements.append(Spacer(1, 0.1 * inch))
        elements.append(Paragraph(src.page_content.replace('\n', '<br/>'), body_style))
        elements.append(Spacer(1, 0.2 * inch))

    doc.build(elements)
    buffer.seek(0)
    return buffer


timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
pdf_path  = f"query_{timestamp}.pdf"

pdf_buffer = create_pdf(
    user_query,
    result["answer"],
    result["context"],
    selected_authors,
    date_range=selected_date_range
)

with open(pdf_path, "wb") as f:
    f.write(pdf_buffer.read())

print(f"PDF saved to: {pdf_path}")